## Entorno

### Dependencias

In [ ]:
!pip install -U -q PyDrive
!pip install unidecode
!pip install ultralytics
!pip install onnx
!pip install imgaug
!pip install albumentations==1.3.0
!pip install opencv-python
!pip install pybboxes
!pip install fiftyone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadat

In [ ]:
import pandas as pd
import re
import unidecode
import codecs
import cv2
import torch
import glob
import locale
from collections import Counter

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from google.colab import drive as dri
from oauth2client.client import GoogleCredentials
from google_drive_downloader import GoogleDriveDownloader as gdd

import sys
import inspect
import os
import zipfile
import shutil
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from ultralytics import YOLO

import seaborn as sns
import matplotlib.pyplot as plt

import fiftyone as fo
import fiftyone.zoo as foz

In [ ]:
locale.getpreferredencoding = lambda: "UTF-8"
locale.getpreferredencoding = lambda x: "UTF-8"

### Autenticacion

In [ ]:
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# dri.mount('/content/drive')

In [ ]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def to_device(data, device):
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return
        len(self.dl)

device = get_default_device()
device

device(type='cuda')

## Dataset

### Descarga

### Reoganizacion

### FiftyOne

In [ ]:
# %%writefile /content/dataset/dataset.yaml

# train: /content/dataset/images/train
# val: /content/dataset/images/val

# nc: 49

# names: [
#     "1O", "1C", "1E", "1B",
#     "2O", "2C", "2E", "2B",
#     "3O", "3C", "3E", "3B",
#     "4O", "4C", "4E", "4B",
#     "5O", "5C", "5E", "5B",
#     "6O", "6C", "6E", "6B",
#     "7O", "7C", "7E", "7B",
#     "8O", "8C", "8E", "8B",
#     "9O", "9C", "9E", "9B",
#     "10O", "10C", "10E", "10B",
#     "11O", "11C", "11E", "11B",
#     "12O", "12C", "12E", "12B",
#     "J"
# ]

In [ ]:
# name = "cartas"
# dataset_dir = "/content/dataset"

# # The splits to load
# splits = ["train", "val"]

# # Load the dataset, using tags to mark the samples in each split
# dataset = fo.Dataset(name)
# for split in splits:
#     dataset.add_dir(
#         dataset_dir=dataset_dir,
#         dataset_type=fo.types.YOLOv5Dataset,
#         split=split,
#         tags=split,
# )

In [ ]:
# session = fo.launch_app(dataset)

## Modelo

### Yolo v8

In [ ]:
model = YOLO("yolov8s.pt")

model.train(data="/content/dataset.yaml", epochs=20, imgsz=640, batch=16, lr0=0.001)
metrics = model.val()
path = model.export(format="onnx")

100%|██████████| 21.5M/21.5M [00:00<00:00, 142MB/s] 


Ultralytics YOLOv8.2.63 🚀 Python-3.10.12 torch-2.3.1+cu121 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/dataset.yaml, epochs=20, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=Tr

100%|██████████| 755k/755k [00:00<00:00, 15.1MB/s]


Overriding model.yaml nc=80 with nc=49

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytic

train: Scanning /content/dataset/labels/train... 12 images, 0 backgrounds, 0 corrupt: 100%|██████████| 12/12 [00:00<00:00, 65.94it/s]

...

       1/20         0G      1.742      5.231      1.724        166        640: 100%|██████████| 1/1 [00:48<00:00, 48.47s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.87s/it]

                   all          2         19     0.0357      0.111     0.0355     0.0332



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/1 [00:39<?, ?it/s]


KeyboardInterrupt: 

### Prediccion

In [ ]:
model.predict("/content/dataset/images/val/A45161_Fabian_Aguirre_12.png", save=True, imgsz=320, conf=0.5)


image 1/1 /content/dataset/images/val/A45161_Fabian_Aguirre_12.png: 256x320 2 12Bs, 93.9ms
Speed: 2.0ms preprocess, 93.9ms inference, 2.6ms postprocess per image at shape (1, 3, 256, 320)
Results saved to runs/detect/train3


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: '1O', 1: '1C', 2: '1E', 3: '1B', 4: '2O', 5: '2C', 6: '2E', 7: '2B', 8: '3O', 9: '3C', 10: '3E', 11: '3B', 12: '4O', 13: '4C', 14: '4E', 15: '4B', 16: '5O', 17: '5C', 18: '5E', 19: '5B', 20: '6O', 21: '6C', 22: '6E', 23: '6B', 24: '7O', 25: '7C', 26: '7E', 27: '7B', 28: '8O', 29: '8C', 30: '8E', 31: '8B', 32: '9O', 33: '9C', 34: '9E', 35: '9B', 36: '10O', 37: '10C', 38: '10E', 39: '10B', 40: '11O', 41: '11C', 42: '11E', 43: '11B', 44: '12O', 45: '12C', 46: '12E', 47: '12B', 48: 'J'}
 obb: None
 orig_img: array([[[ 8, 26, 49],
         [ 8, 26, 49],
         [10, 26, 49],
         ...,
         [19, 38, 73],
         [21, 40, 75],
         [22, 41, 76]],
 
        [[ 8, 26, 49],
         [ 8, 26, 49],
         [ 9, 25, 48],
         ...,
         [17, 36, 71],
         [17, 36, 71],
         [18, 37, 72]],
 
        [[ 9, 26, 47],
      